# LoveDA: baseline phần 1/2 — 10 model đầu

Notebook này chạy riêng nhóm 1/2. Cơ chế dò batch size tối đa, bắt CUDA OOM, giảm batch và tiếp tục model kế tiếp giống notebook đầy đủ.


In [ ]:
# Cấu hình: sửa trước khi Run All
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/alienxc137/loveda-satellite-images-semantic-segmentation")
TRAIN_DIR, VAL_DIR = DATA_ROOT / "Train", DATA_ROOT / "Val"
OUTPUT_ROOT = Path("/kaggle/working/loveda_baseline_runs_part1")

EPOCHS = 10                 # 5-10 để ưu tiên số model trong một phiên; tăng khi cần
CROP_SIZE = 512
MAX_BATCH_SIZE = 64         # chỉ là trần; notebook tự tìm batch tối đa theo GPU
VAL_BATCH_SIZE = 1
NUM_WORKERS = 2
LR, WEIGHT_DECAY = 6e-5, 1e-2
AMP, PRETRAINED = True, False
SEED, VAL_EVERY = 42, 2
TIME_BUDGET_HOURS = 11.5    # None để tắt

MODELS = [
    "pspnet",
    "deeplabv3plus",
    "danet",
    "semantic_fpn",
    "farseg",
    "ocrnet",
    "lanet",
    "isnet",
    "segmenter",
    "swin_upernet"
]


In [ ]:
# Kiểm tra dataset và lấy baseline code
import subprocess, sys

assert TRAIN_DIR.is_dir(), f"Không tìm thấy {TRAIN_DIR}"
assert VAL_DIR.is_dir(), f"Không tìm thấy {VAL_DIR}"

candidates = [
    Path("/kaggle/working/LandCover/baseline_model"),
    Path("/kaggle/working/baseline_model"),
    Path.cwd() / "baseline_model",
]
BASELINE_ROOT = next((p for p in candidates if (p / "baseline_suite").is_dir()), None)
if BASELINE_ROOT is None:
    clone_root = Path("/kaggle/working/LandCover")
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/TuanHungGiang/LandCover.git", str(clone_root)],
            check=True,
        )
        BASELINE_ROOT = clone_root / "baseline_model"
    except Exception as exc:
        raise RuntimeError(
            "Không lấy được baseline code. Hãy bật Internet hoặc gắn repository LandCover "
            "thành Kaggle Dataset để có thư mục baseline_model."
        ) from exc

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm>=0.9", "tqdm>=4.65"], check=True)
sys.path.insert(0, str(BASELINE_ROOT))
print("Baseline:", BASELINE_ROOT)
print("Train:", TRAIN_DIR)
print("Val:", VAL_DIR)


In [ ]:
import gc, json, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from baseline_suite.data import LOVEDA_CLASSES, LoveDADataset
from baseline_suite.metrics import SegmentationMetrics
from baseline_suite.models import MODEL_NAMES, build_model, model_info

assert torch.cuda.is_available(), "Hãy bật GPU Accelerator trong Kaggle."
assert not (set(MODELS) - set(MODEL_NAMES)), "MODELS chứa tên không hợp lệ."

device = torch.device("cuda")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SESSION_START = time.monotonic()
DEADLINE = None if TIME_BUDGET_HOURS is None else SESSION_START + TIME_BUDGET_HOURS * 3600

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

train_ds = LoveDADataset(DATA_ROOT, "Train", CROP_SIZE, training=True, seed=SEED)
val_ds = LoveDADataset(DATA_ROOT, "Val", CROP_SIZE, training=False, seed=SEED)
print(torch.cuda.get_device_name(0), "| Train:", len(train_ds), "| Val:", len(val_ds))


In [ ]:
def is_oom(exc):
    return isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in str(exc).lower()

def clear_vram():
    gc.collect()
    try:
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    except Exception:
        pass

def amp_context():
    return torch.autocast("cuda", dtype=torch.float16, enabled=AMP)

def loader(ds, bs, training):
    return DataLoader(
        ds, batch_size=bs, shuffle=training, num_workers=NUM_WORKERS,
        pin_memory=True, drop_last=training, persistent_workers=NUM_WORKERS > 0,
    )

def sliding_predict(model, image):
    b, _, h, w = image.shape
    if h == CROP_SIZE and w == CROP_SIZE:
        return model(image)
    logits = image.new_zeros((b, 7, h, w))
    counts = image.new_zeros((b, 1, h, w))
    for top in range(0, h, CROP_SIZE):
        for left in range(0, w, CROP_SIZE):
            bottom, right = min(top + CROP_SIZE, h), min(left + CROP_SIZE, w)
            tile = image[:, :, bottom-CROP_SIZE:bottom, right-CROP_SIZE:right]
            logits[:, :, bottom-CROP_SIZE:bottom, right-CROP_SIZE:right] += model(tile)
            counts[:, :, bottom-CROP_SIZE:bottom, right-CROP_SIZE:right] += 1
    return logits / counts.clamp_min(1)

@torch.inference_mode()
def evaluate(model):
    model.eval()
    metrics, losses = SegmentationMetrics(7, 255), []
    for batch in tqdm(loader(val_ds, VAL_BATCH_SIZE, False), desc="LoveDA Val", leave=False):
        image = batch["image"].to(device, non_blocking=True)
        target = batch["mask"].to(device, non_blocking=True)
        with amp_context():
            logits = sliding_predict(model, image)
            loss = F.cross_entropy(logits, target, ignore_index=255)
        metrics.update(logits, target)
        losses.append(loss.item())
    result = metrics.compute()
    result.update(loss=float(np.mean(losses)), class_names=list(LOVEDA_CLASSES))
    return result


In [ ]:
def batch_fits(model_name, bs):
    # Có optimizer.step(): phép đo bao gồm VRAM của AdamW state, không chỉ forward.
    model = opt = scaler = dl = batch = image = target = logits = loss = None
    try:
        clear_vram()
        torch.cuda.reset_peak_memory_stats()
        dl = loader(train_ds, bs, True)
        batch = next(iter(dl))
        model = build_model(model_name, 7, pretrained=PRETRAINED, image_size=CROP_SIZE).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scaler = torch.amp.GradScaler("cuda", enabled=AMP)
        model.train()
        image = batch["image"].to(device, non_blocking=True)
        target = batch["mask"].to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with amp_context():
            logits = model(image)
            loss = F.cross_entropy(logits, target, ignore_index=255)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); torch.cuda.synchronize()
        return True, torch.cuda.max_memory_allocated() / 2**30
    except Exception as exc:
        if is_oom(exc):
            return False, None
        raise
    finally:
        del model, opt, scaler, dl, batch, image, target, logits, loss
        clear_vram()

def find_max_batch(model_name):
    # Exponential search rồi binary search: trả về batch nguyên lớn nhất <= MAX_BATCH_SIZE.
    cap, low, bad, size = min(MAX_BATCH_SIZE, len(train_ds)), 0, None, 1
    while size <= cap:
        ok, peak = batch_fits(model_name, size)
        print(f"  probe bs={size:<3} -> {'OK' if ok else 'OOM'}" + (f" ({peak:.2f} GiB)" if peak else ""))
        if not ok:
            bad = size
            break
        low, size = size, size * 2
    if low == 0:
        return 0
    high = cap if bad is None else bad - 1
    while low < high:
        mid = (low + high + 1) // 2
        ok, peak = batch_fits(model_name, mid)
        print(f"  probe bs={mid:<3} -> {'OK' if ok else 'OOM'}" + (f" ({peak:.2f} GiB)" if peak else ""))
        if ok: low = mid
        else: high = mid - 1
    return low


In [ ]:
class TrainingOOM(RuntimeError):
    pass

def save_best(path, model, epoch, metrics, bs):
    torch.save({"epoch": epoch, "model": model.state_dict(), "metrics": metrics, "batch_size": bs}, path)

def train_one(model_name, bs):
    run_dir = OUTPUT_ROOT / model_name
    run_dir.mkdir(parents=True, exist_ok=True)
    model = opt = sched = scaler = train_dl = None
    try:
        model = build_model(model_name, 7, pretrained=PRETRAINED, image_size=CROP_SIZE).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS, eta_min=LR * .01)
        scaler = torch.amp.GradScaler("cuda", enabled=AMP)
        train_dl = loader(train_ds, bs, True)
        history, best = [], -1.0
        for epoch in range(1, EPOCHS + 1):
            if DEADLINE is not None and time.monotonic() >= DEADLINE:
                return {"status": "time_budget", "batch_size": bs, "best_miou": best}
            model.train(); running = 0.0
            bar = tqdm(train_dl, desc=f"{model_name} {epoch}/{EPOCHS}", leave=False)
            try:
                for step, batch in enumerate(bar, 1):
                    image = batch["image"].to(device, non_blocking=True)
                    target = batch["mask"].to(device, non_blocking=True)
                    opt.zero_grad(set_to_none=True)
                    with amp_context():
                        logits = model(image)
                        loss = F.cross_entropy(logits, target, ignore_index=255)
                    scaler.scale(loss).backward()
                    scaler.step(opt); scaler.update()
                    running += loss.item()
                    bar.set_postfix(loss=f"{running/step:.4f}", bs=bs)
                    del image, target, logits, loss
            except Exception as exc:
                if is_oom(exc): raise TrainingOOM() from None
                raise
            sched.step()
            rec = {"epoch": epoch, "train_loss": running / max(1, len(train_dl))}
            if epoch % VAL_EVERY == 0 or epoch == EPOCHS:
                try:
                    val = evaluate(model)
                except Exception as exc:
                    if is_oom(exc): raise TrainingOOM() from None
                    raise
                rec.update(val)
                if float(val["miou"]) > best:
                    best = float(val["miou"])
                    save_best(run_dir / "best.pt", model, epoch, val, bs)
            history.append(rec)
            (run_dir / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
            print(f"  epoch={epoch} loss={rec['train_loss']:.4f}" + (f" mIoU={rec['miou']:.4f}" if "miou" in rec else ""))
        return {"status": "completed", "batch_size": bs, "best_miou": best}
    finally:
        del model, opt, sched, scaler, train_dl
        clear_vram()


In [ ]:
# Chạy tuần tự. Không có exception theo model nào được thoát khỏi cell này.
summary_file = OUTPUT_ROOT / "summary.csv"
state_file = OUTPUT_ROOT / "run_state.json"
rows = pd.read_csv(summary_file).to_dict("records") if summary_file.exists() else []
completed = {r["model"] for r in rows if r.get("status") == "completed"}

for model_name in MODELS:
    if model_name in completed:
        print(f"\n[SKIP] {model_name}: đã hoàn tất")
        continue
    if DEADLINE is not None and time.monotonic() >= DEADLINE:
        print("\nĐã chạm time budget; dừng sạch để Kaggle lưu output.")
        break
    print(f"\n=== {model_name} | {model_info(model_name)} ===")
    started = time.monotonic()
    row = {"model": model_name, "status": "pending", "batch_size": 0, "best_miou": -1.0}
    try:
        bs = find_max_batch(model_name)
        if bs < 1:
            row["status"] = "oom_at_batch_1"
            print("  Batch 1 vẫn OOM; chuyển model kế tiếp.")
        else:
            # Probe có thể qua nhưng batch khác vẫn OOM: giảm 1 và khởi động model lại.
            while bs >= 1:
                try:
                    print(f"  Train với batch_size={bs}")
                    row.update(train_one(model_name, bs))
                    break
                except TrainingOOM:
                    print(f"  OOM trong train; giảm batch {bs} -> {bs-1}.")
                    bs -= 1
                    clear_vram()
            if bs < 1:
                row["status"] = "oom_at_batch_1"
    except Exception as exc:
        # Ghi thông báo gọn, không traceback và không làm dừng model sau.
        row["status"] = "failed_non_oom"
        row["message"] = f"{type(exc).__name__}: {str(exc)[:300]}"
        print("  Lỗi không phải OOM; đã ghi summary và chuyển tiếp:", row["message"])
    finally:
        row["minutes"] = round((time.monotonic() - started) / 60, 2)
        rows = [r for r in rows if r.get("model") != model_name] + [row]
        pd.DataFrame(rows).to_csv(summary_file, index=False)
        state_file.write_text(json.dumps({"models_requested": MODELS, "results": rows}, indent=2), encoding="utf-8")
        clear_vram()

display(pd.DataFrame(rows))
print("Kết quả:", OUTPUT_ROOT)


## Lưu ý

- Để train được nhiều model trong một phiên Kaggle, mặc định dùng 10 epoch. Có thể chia `MODELS` thành nhiều nhóm hoặc tăng `EPOCHS` cho lần chạy chính thức.
- Probe tính cả forward, backward và `AdamW.step()`. Nếu train thật vẫn OOM, notebook tự giảm batch một đơn vị và thử lại từ đầu.
- `PRETRAINED=False` không cần tải weights. Việc clone repository lần đầu cần Internet; nếu tắt Internet, hãy gắn repository làm Kaggle Dataset.
- Full LoveDA Val được dùng định kỳ và ở epoch cuối; LoveDA Test không được dùng.
